# Import libs

In [1]:
import os
import sys

project_root = "Incremental-Learning-MycroftMinds"
cwd = os.path.abspath(os.getcwd())
while True:
    if os.path.basename(cwd) == project_root:
        if cwd not in sys.path:
            sys.path.insert(0, cwd)
        break
    parent = os.path.dirname(cwd)
    if parent == cwd:
        raise FileNotFoundError(f"Project root '{project_root}' not found in any parent directory.")
    cwd = parent

In [2]:
from src.data import dataloader
from src.data.io import load_dataset, save_dataset
from src.utils.memory import print_memory_stats
from src.trees import fastodt
from src.data.preprocessing import read_second_batch
import numpy as np
import pandas as pd
import time

# Commercial data splits

In [4]:
data_dir = '../../data'

lag_args = {
    'lag_1h':'1H',  # all based on PACF
    'lag_2h':'2H',
    'lag_3h':'3H',
    'lag_24h':'24H',
    #'lag_168h':'168H' # 24h*7 = 168h, one week ago
    } 

# list feature names
columns_to_add_lag = [
            'consumption'
        ]

# target
target_feature = 'consumption'
batch='second_batch'
location='commercial'
device_list = ['sub1', 'sub2', 'sub3']


# Load data using the optimized function
# First try the optimized version, if not available fall back to regular version
train, valid, test = dataloader.get_train_test_sets(
    data_dir=data_dir,
    lag_freqs=lag_args,
    columns_to_add_lag=columns_to_add_lag,
    target_feature=target_feature,
    device_list=device_list,
    walk_forward_sets=True,
    train_days=28,
    valid_days=28,
    optimize_memory=True,  # Enable memory optimization
    save=True,  # Save processed data
    intermediate_prefix='sb_com_hour_wfv',  # Prefix for saved files
    time_agg='1h',  # Time aggregation frequency
    new_data=batch,
    location=location
)

[INFO] Aggregating data by: 1h
input_features:  ['time', 'device_id', 'hour', 'weekday', 'month', 'consumption_lag_1h', 'consumption_lag_2h', 'consumption_lag_3h', 'consumption_lag_24h']

Processing site: sub1
Processing site: sub2
Processing site: sub3
Overall data time range:  2024-06-06 22:00:00+02:00 2024-08-29 11:00:00+02:00

Test data time range:  2024-08-12 14:00:00+02:00 2024-08-29 11:00:00+02:00

(Train + Valid) data range:  2024-06-06 22:00:00+02:00 2024-08-12 12:00:00+02:00

Split 1:
  Train window: 2024-06-06 22:00:00+02:00 to 2024-07-04 21:00:00+02:00
  Valid window: 2024-07-04 22:00:00+02:00 to 2024-08-01 21:00:00+02:00
  Train samples: 2016, Valid samples: 2016

Split 2:
  Train window: 2024-06-06 22:00:00+02:00 to 2024-08-01 21:00:00+02:00
  Valid window: 2024-08-01 22:00:00+02:00 to 2024-08-12 12:00:00+02:00
  Train samples: 4032, Valid samples: 765
Memory usage after optimization: Train=0.46MB, Valid=0.21MB, Test=0.09MB
[INFO] Saving train data to ../../data\second_ba

## Check for missing datetimes

In [ ]:
# Concatenate all valid splits
valid_arrays = [split[0] for split in valid]
valid_concat = np.concatenate(valid_arrays, axis=0)

# Extract the 'time' column (assumed to be the first column)
all_times = pd.Series(valid_concat[:, 0])

# Convert to pandas datetime if not already
all_times = pd.to_datetime(all_times)

# Create a complete time index with 15min frequency
full_range = pd.date_range(start=all_times.min(), end=all_times.max(), freq='1h', tz=all_times.dt.tz)

# Find missing datetimes
missing_times = full_range.difference(all_times)

print(f"Total timestamps in full range: {len(full_range)}")
print(f"Number of missing timestamps: {len(missing_times)}")
print(missing_times)

# Industrial data splits

In [5]:
data_dir = '../../data/'

lag_args = {
    'lag_1h':'1H',  # all based on PACF
    'lag_2h':'2H',
    'lag_24h':'24H',
    'lag_25h':'25H',
    #'lag_168h':'168H' # 24h*7 = 168h, one week ago
    } 

# list feature names
columns_to_add_lag = [
            'consumption'
        ]

# target
target_feature = 'consumption'
batch='second_batch'
location='industrial'


# Load data using the optimized function
# First try the optimized version, if not available fall back to regular version
train, valid, test = dataloader.get_train_test_sets(
    data_dir=data_dir,
    lag_freqs=lag_args,
    columns_to_add_lag=columns_to_add_lag,
    target_feature=target_feature,
    device_list=None,
    walk_forward_sets=True,
    train_days=28,
    valid_days=28,
    optimize_memory=True,  # Enable memory optimization
    save=True,  # Save processed data for faster future loading
    intermediate_prefix='sb_ind_hour_wfv',  # Prefix for saved files
    time_agg='1H',  # Time aggregation frequency
    new_data=batch,  # 'first_batch' or 'second_batch' (defaults 'second_batch')
    location=location  # to be used with second batch. specify 'industrial' or 'commercial' (only valid for second batch data)
)

[INFO] Aggregating data by: 1H
input_features:  ['time', 'device_id', 'hour', 'weekday', 'month', 'consumption_lag_1h', 'consumption_lag_2h', 'consumption_lag_24h', 'consumption_lag_25h']

Processing site: site1
Processing site: site2
Processing site: site3
Processing site: site4
Processing site: site5
Processing site: site8
Processing site: site9
Overall data time range:  2023-02-27 02:00:00+01:00 2025-04-24 19:00:00+02:00

Test data time range:  2025-02-23 18:00:00+01:00 2025-04-24 19:00:00+02:00

(Train + Valid) data range:  2023-02-27 02:00:00+01:00 2025-02-23 17:00:00+01:00

Split 1:
  Train window: 2023-02-27 02:00:00+01:00 to 2023-03-27 02:00:00+02:00
  Valid window: 2023-03-27 03:00:00+02:00 to 2023-04-24 02:00:00+02:00
  Train samples: 1346, Valid samples: 1344

Split 2:
  Train window: 2023-02-27 02:00:00+01:00 to 2023-04-24 02:00:00+02:00
  Valid window: 2023-04-24 03:00:00+02:00 to 2023-05-22 02:00:00+02:00
  Train samples: 2690, Valid samples: 1344

Split 3:
  Train window

## Check for missing datetimes

In [6]:
# Concatenate all valid splits
valid_arrays = [split[0] for split in valid]
valid_concat = np.concatenate(valid_arrays, axis=0)

# Extract the 'time' column (assumed to be the first column)
all_times = pd.Series(valid_concat[:, 0])

# Convert to pandas datetime if not already
all_times = pd.to_datetime(all_times)

# Create a complete time index with 15min frequency
full_range = pd.date_range(start=all_times.min(), end=all_times.max(), freq='1h', tz=all_times.dt.tz)

# Find missing datetimes
missing_times = full_range.difference(all_times)

print(f"Total timestamps in full range: {len(full_range)}")
print(f"Number of missing timestamps: {len(missing_times)}")
print(missing_times)

Total timestamps in full range: 16792
Number of missing timestamps: 6
DatetimeIndex(['2023-10-29 02:00:00+02:00', '2023-10-29 02:00:00+01:00',
               '2023-10-29 03:00:00+01:00', '2024-10-27 02:00:00+02:00',
               '2024-10-27 02:00:00+01:00', '2024-10-27 03:00:00+01:00'],
              dtype='datetime64[ns, Europe/Prague]', freq=None)
